# Assignment 4: CIFAR-10 Diffusion and Energy Models
Liuyang Li

Select a GPU runtime for longer training. This notebook contains its own source. The supplied small CPU checkpoints demonstrate the workflow; visual quality and convergence require separate evaluation. The longer runs below have not already been executed.

In [ ]:
import torch, torchvision, sys, json, os, subprocess
from pathlib import Path
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.is_available())


In [ ]:
WORK = Path("/content/assignment4") if Path("/content").exists() else Path.cwd()/"assignment4_training"
WORK.mkdir(parents=True,exist_ok=True)
SOURCE_FILES = {'app/image_generation/models.py': '"""Time-conditioned U-Net and scalar energy CNN for native 32x32 CIFAR-10."""\n\nimport math\nimport torch\nfrom torch import nn\n\n\nclass SinusoidalTimeEmbedding(nn.Module):\n    def __init__(self, dim=128, max_period=10000):\n        super().__init__()\n        if dim < 2 or dim % 2:\n            raise ValueError("Time dimension must be positive and even.")\n        self.register_buffer(\n            "frequencies",\n            torch.exp(-math.log(max_period) * torch.arange(dim // 2) / (dim // 2)),\n        )\n\n    def forward(self, t):\n        angles = t.float()[:, None] * self.frequencies[None]\n        return torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(1)\n\n\nclass TimeResidualBlock(nn.Module):\n    def __init__(self, cin, cout, time_dim):\n        super().__init__()\n        self.first = nn.Sequential(\n            nn.GroupNorm(8, cin), nn.SiLU(), nn.Conv2d(cin, cout, 3, padding=1)\n        )\n        self.time = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, cout))\n        self.second = nn.Sequential(\n            nn.GroupNorm(8, cout), nn.SiLU(), nn.Conv2d(cout, cout, 3, padding=1)\n        )\n        self.skip = nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1)\n\n    def forward(self, x, time):\n        return self.skip(x) + self.second(\n            self.first(x) + self.time(time)[:, :, None, None]\n        )\n\n\nclass CIFARUNet(nn.Module):\n    def __init__(self, base_channels=32, time_dim=128):\n        super().__init__()\n        if base_channels < 8 or base_channels % 8:\n            raise ValueError("base_channels must be a positive multiple of eight.")\n        b = base_channels\n        self.time = nn.Sequential(\n            SinusoidalTimeEmbedding(time_dim),\n            nn.Linear(time_dim, time_dim),\n            nn.SiLU(),\n            nn.Linear(time_dim, time_dim),\n        )\n        self.input = nn.Conv2d(3, b, 3, padding=1)\n        self.enc0 = TimeResidualBlock(b, b, time_dim)\n        self.down0 = nn.Conv2d(b, 2 * b, 4, stride=2, padding=1)\n        self.enc1 = TimeResidualBlock(2 * b, 2 * b, time_dim)\n        self.down1 = nn.Conv2d(2 * b, 4 * b, 4, stride=2, padding=1)\n        self.enc2 = TimeResidualBlock(4 * b, 4 * b, time_dim)\n        self.down2 = nn.Conv2d(4 * b, 4 * b, 4, stride=2, padding=1)\n        self.mid1 = TimeResidualBlock(4 * b, 4 * b, time_dim)\n        self.mid2 = TimeResidualBlock(4 * b, 4 * b, time_dim)\n        self.up2 = nn.ConvTranspose2d(4 * b, 4 * b, 4, stride=2, padding=1)\n        self.dec2 = TimeResidualBlock(8 * b, 4 * b, time_dim)\n        self.up1 = nn.ConvTranspose2d(4 * b, 2 * b, 4, stride=2, padding=1)\n        self.dec1 = TimeResidualBlock(4 * b, 2 * b, time_dim)\n        self.up0 = nn.ConvTranspose2d(2 * b, b, 4, stride=2, padding=1)\n        self.dec0 = TimeResidualBlock(2 * b, b, time_dim)\n        self.output = nn.Sequential(\n            nn.GroupNorm(8, b), nn.SiLU(), nn.Conv2d(b, 3, 3, padding=1)\n        )\n\n    def forward(self, x, t):\n        time = self.time(t)\n        h0 = self.enc0(self.input(x), time)\n        h1 = self.enc1(self.down0(h0), time)\n        h2 = self.enc2(self.down1(h1), time)\n        h = self.mid2(self.mid1(self.down2(h2), time), time)\n        h = self.dec2(torch.cat((self.up2(h), h2), 1), time)\n        h = self.dec1(torch.cat((self.up1(h), h1), 1), time)\n        h = self.dec0(torch.cat((self.up0(h), h0), 1), time)\n        return self.output(h)\n\n\nclass CIFAREnergyModel(nn.Module):\n    def __init__(self, base_channels=32):\n        super().__init__()\n        b = base_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(3, b, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(b, 2 * b, 4, 2, 1),\n            nn.SiLU(),\n            nn.Conv2d(2 * b, 4 * b, 4, 2, 1),\n            nn.SiLU(),\n            nn.Conv2d(4 * b, 4 * b, 4, 2, 1),\n            nn.SiLU(),\n            nn.Flatten(),\n            nn.Linear(4 * b * 4 * 4, 1),\n        )\n\n    def forward(self, x):\n        return self.net(x).squeeze(1)\n', 'app/image_generation/diffusion.py': '"""DDPM noise prediction, cosine schedule, and ancestral reverse sampling."""\n\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass DDPMSchedule(nn.Module):\n    def __init__(self, timesteps=200):\n        super().__init__()\n        if timesteps < 2:\n            raise ValueError("At least two timesteps are required.")\n        self.timesteps = timesteps\n        t = torch.linspace(0, timesteps, timesteps + 1, dtype=torch.float64) / timesteps\n        cumulative = torch.cos((t + 0.008) / 1.008 * math.pi / 2).square()\n        cumulative = cumulative / cumulative[0]\n        beta = (1 - cumulative[1:] / cumulative[:-1]).clamp(1e-8, 0.999).float()\n        alpha = 1 - beta\n        abar = torch.cumprod(alpha, 0)\n        previous = F.pad(abar[:-1], (1, 0), value=1.0)\n        for name, value in {\n            "betas": beta,\n            "alpha_bar": abar,\n            "posterior_variance": beta * (1 - previous) / (1 - abar),\n            "mean_x0": beta * previous.sqrt() / (1 - abar),\n            "mean_xt": (1 - previous) * alpha.sqrt() / (1 - abar),\n        }.items():\n            self.register_buffer(name, value)\n\n    def add_noise(self, clean, t, noise):\n        a = self.alpha_bar[t][:, None, None, None]\n        return a.sqrt() * clean + (1 - a).sqrt() * noise\n\n    def training_loss(self, model, clean):\n        t = torch.randint(self.timesteps, (len(clean),), device=clean.device)\n        noise = torch.randn_like(clean)\n        return F.mse_loss(model(self.add_noise(clean, t, noise), t), noise)\n\n    @torch.no_grad()\n    def sample(self, model, count, generator):\n        device = self.betas.device\n        image = torch.randn((count, 3, 32, 32), generator=generator, device=device)\n        for step in reversed(range(self.timesteps)):\n            t = torch.full((count,), step, device=device, dtype=torch.long)\n            epsilon = model(image, t)\n            a = self.alpha_bar[step]\n            x0 = ((image - (1 - a).sqrt() * epsilon) / a.sqrt()).clamp(-1, 1)\n            mean = self.mean_x0[step] * x0 + self.mean_xt[step] * image\n            if step:\n                noise = torch.randn(image.shape, generator=generator, device=device)\n                image = mean + self.posterior_variance[step].sqrt() * noise\n            else:\n                image = mean\n        return image.clamp(-1, 1)\n', 'app/image_generation/energy.py': '"""Sampling changes pixels; the later training backward changes weights."""\n\nfrom contextlib import contextmanager\nimport torch\n\n\n@contextmanager\ndef frozen_parameters(model):\n    flags = [p.requires_grad for p in model.parameters()]\n    try:\n        for p in model.parameters():\n            p.requires_grad_(False)\n        yield\n    finally:\n        for p, flag in zip(model.parameters(), flags):\n            p.requires_grad_(flag)\n\n\ndef langevin_sample(\n    model, initial, steps=60, step_size=10.0, noise_std=0.005, generator=None\n):\n    """Practical short-run x <- clamp(x-eta*grad(E)+sigma*z).\n\n    Gradient clipping and the small noise setting make this approximate synthesis,\n    not exact equilibrium sampling. Ideal ULA for exp(-E) uses sigma=sqrt(2*eta).\n    """\n    if steps < 1 or step_size <= 0 or noise_std < 0:\n        raise ValueError("Invalid sampler settings.")\n    image = initial.detach().clone()\n    for _ in range(steps):\n        with torch.enable_grad():\n            image = image.detach().requires_grad_(True)\n            (gradient,) = torch.autograd.grad(\n                model(image).sum(), image, create_graph=False\n            )\n        with torch.no_grad():\n            noise = torch.randn(image.shape, device=image.device, generator=generator)\n            image = (\n                image - step_size * gradient.clamp(-0.03, 0.03) + noise_std * noise\n            ).clamp(-1, 1)\n    return image.detach()\n\n\nclass ReplayBuffer:\n    def __init__(self, size=1024):\n        self.images = torch.empty(size, 3, 32, 32).uniform_(-1, 1)\n\n    def sample(self, count, device):\n        old = self.images[torch.randint(len(self.images), (count,))].to(device)\n        fresh = torch.empty_like(old).uniform_(-1, 1)\n        restart = torch.rand(count, 1, 1, 1, device=device) < 0.05\n        return torch.where(restart, fresh, old)\n\n    def update(self, images):\n        indices = torch.randperm(len(self.images))[: len(images)]\n        self.images[indices] = images[: len(indices)].detach().cpu()\n', 'training/train_images.py': '"""CIFAR-10 training shared by the diffusion and energy command-line entry points."""\n\nimport argparse, copy, json, random, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\nfrom torchvision import datasets, transforms\nfrom torchvision.utils import save_image\nfrom app.image_generation.models import CIFARUNet, CIFAREnergyModel\nfrom app.image_generation.diffusion import DDPMSchedule\nfrom app.image_generation.energy import ReplayBuffer, frozen_parameters, langevin_sample\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef main(kind):\n    p = argparse.ArgumentParser()\n    p.add_argument("--epochs", type=int, default=50 if kind == "diffusion" else 30)\n    p.add_argument("--base-channels", type=int, default=32)\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--timesteps", type=int, default=200)\n    p.add_argument("--langevin-steps", type=int, default=20)\n    p.add_argument("--step-size", type=float, default=10.0)\n    p.add_argument("--noise-std", type=float, default=0.005)\n    p.add_argument("--lr", type=float, default=2e-4 if kind == "diffusion" else 1e-4)\n    p.add_argument("--data-dir", default=str(ROOT / "data"))\n    p.add_argument("--prepared-data", default=None)\n    p.add_argument("--output", default=str(ROOT / f"models/cifar10_{kind}.pt"))\n    p.add_argument("--device", default="auto")\n    p.add_argument("--threads", type=int, default=2)\n    p.add_argument("--seed", type=int, default=42 if kind == "diffusion" else 43)\n    p.add_argument("--resume", default=None)\n    p.add_argument(\n        "--max-steps",\n        type=int,\n        default=0,\n        help="Optional total-step limit for smoke runs",\n    )\n    a = p.parse_args()\n    if min(a.epochs, a.batch_size, a.threads) < 1:\n        p.error("epochs, batch size and threads must be positive")\n    random.seed(a.seed)\n    np.random.seed(a.seed)\n    torch.manual_seed(a.seed)\n    torch.set_num_threads(a.threads)\n    device = torch.device(\n        ("cuda" if torch.cuda.is_available() else "cpu")\n        if a.device == "auto"\n        else a.device\n    )\n    transform = transforms.Compose(\n        [\n            transforms.RandomHorizontalFlip(),\n            transforms.ToTensor(),\n            transforms.Normalize((0.5,) * 3, (0.5,) * 3),\n        ]\n    )\n    if a.prepared_data:\n        from app.cifar_dataset import PreparedCIFAR10\n\n        data = PreparedCIFAR10(Path(a.prepared_data), train=True, transform=transform)\n    else:\n        data = datasets.CIFAR10(\n            a.data_dir, train=True, download=True, transform=transform\n        )\n    loader = DataLoader(\n        data,\n        batch_size=a.batch_size,\n        shuffle=True,\n        num_workers=0,\n        generator=torch.Generator().manual_seed(a.seed),\n    )\n    model = (\n        CIFARUNet(a.base_channels)\n        if kind == "diffusion"\n        else CIFAREnergyModel(a.base_channels)\n    ).to(device)\n    optimizer = (\n        torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)\n        if kind == "diffusion"\n        else torch.optim.Adam(model.parameters(), lr=a.lr, betas=(0.0, 0.999))\n    )\n    ema = (\n        copy.deepcopy(model).eval().requires_grad_(False)\n        if kind == "diffusion"\n        else None\n    )\n    schedule = DDPMSchedule(a.timesteps).to(device) if kind == "diffusion" else None\n    buffer = ReplayBuffer()\n    steps, seen, start_epoch, history = 0, 0, 1, []\n    config = (\n        {"base_channels": a.base_channels, "time_dim": 128, "timesteps": a.timesteps}\n        if kind == "diffusion"\n        else {\n            "base_channels": a.base_channels,\n            "langevin_steps": a.langevin_steps,\n            "step_size": a.step_size,\n            "noise_std": a.noise_std,\n        }\n    )\n    if a.resume:\n        ckpt = torch.load(a.resume, map_location=device, weights_only=True)\n        if ckpt["config"] != config:\n            raise ValueError("Resume architecture and sampler config must match.")\n        model.load_state_dict(ckpt["model_state_dict"])\n        optimizer.load_state_dict(ckpt["optimizer_state_dict"])\n        if ema is not None:\n            ema.load_state_dict(ckpt["ema_state_dict"])\n        if "replay_buffer" in ckpt:\n            buffer.images = ckpt["replay_buffer"].cpu()\n        steps, seen, start_epoch = ckpt["steps"], ckpt["images_seen"], ckpt["epoch"] + 1\n        history = ckpt.get("history", [])\n    start = time.monotonic()\n    for epoch in range(start_epoch, start_epoch + a.epochs):\n        model.train()\n        total, real_total, negative_total, count = 0.0, 0.0, 0.0, 0\n        for images, _ in loader:\n            images = images.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            if kind == "diffusion":\n                loss = schedule.training_loss(model, images)\n            else:\n                real = (images + 0.005 * torch.randn_like(images)).clamp(-1, 1)\n                with frozen_parameters(model):\n                    negative = langevin_sample(\n                        model,\n                        buffer.sample(len(real), device),\n                        a.langevin_steps,\n                        a.step_size,\n                        a.noise_std,\n                    )\n                buffer.update(negative)\n                er, en = model(real), model(negative.detach())\n                loss = (\n                    er.mean()\n                    - en.mean()\n                    + 0.1 * (er.square().mean() + en.square().mean())\n                )\n                real_total += er.mean().item() * len(images)\n                negative_total += en.mean().item() * len(images)\n            if not torch.isfinite(loss):\n                raise RuntimeError("Non-finite training loss")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(\n                model.parameters(), 1.0 if kind == "diffusion" else 10.0\n            )\n            optimizer.step()\n            if ema is not None:\n                with torch.no_grad():\n                    for avg, param in zip(ema.parameters(), model.parameters()):\n                        avg.lerp_(param, 1 - min(0.999, (1 + steps) / (10 + steps)))\n            steps += 1\n            seen += len(images)\n            count += len(images)\n            total += loss.item() * len(images)\n            if steps % 50 == 0:\n                print(\n                    f"{kind} epoch={epoch} step={steps} loss={loss.item():.6f} seconds={time.monotonic()-start:.1f}",\n                    flush=True,\n                )\n            if a.max_steps and steps >= a.max_steps:\n                break\n        record = {\n            "epoch": epoch,\n            "steps": steps,\n            "images_seen": seen,\n            "loss": total / count,\n            "complete_epoch": count == len(data),\n            "elapsed_seconds_this_run": time.monotonic() - start,\n        }\n        if kind == "diffusion":\n            record["noise_mse"] = total / count\n        else:\n            record.update(\n                real_energy=real_total / count, negative_energy=negative_total / count\n            )\n        history.append(record)\n        payload = {\n            "format_version": 1,\n            "model_type": "ddpm" if kind == "diffusion" else "ebm",\n            "dataset": "CIFAR10",\n            "image_shape": [3, 32, 32],\n            "epoch": epoch,\n            "steps": steps,\n            "images_seen": seen,\n            "train_dataset_size": len(data),\n            "config": config,\n            "model_state_dict": model.state_dict(),\n            "optimizer_state_dict": optimizer.state_dict(),\n            "history": history,\n            "seed": a.seed,\n        }\n        if ema is not None:\n            payload["ema_state_dict"] = ema.state_dict()\n        else:\n            payload["replay_buffer"] = buffer.images\n        output = Path(a.output)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        temporary = output.with_suffix(".tmp")\n        torch.save(payload, temporary)\n        temporary.replace(output)\n        results = ROOT / "results/assignment4"\n        results.mkdir(parents=True, exist_ok=True)\n        (results / f"{kind}_training.json").write_text(\n            json.dumps(history, indent=2) + "\\n"\n        )\n        print(record, flush=True)\n        if a.max_steps and steps >= a.max_steps:\n            break\n    generator = torch.Generator(device=device).manual_seed(123)\n    model.eval().requires_grad_(False)\n    if kind == "diffusion":\n        samples = schedule.sample(ema, 16, generator)\n    else:\n        initial = (\n            torch.rand((16, 3, 32, 32), device=device, generator=generator) * 2 - 1\n        )\n        samples = langevin_sample(\n            model, initial, 60, a.step_size, a.noise_std, generator\n        )\n    save_image(\n        (samples.cpu() + 1) / 2,\n        ROOT / f"results/assignment4/{kind}_samples.png",\n        nrow=4,\n    )\n', 'training/train_diffusion.py': 'from training.train_images import main\n\nif __name__ == "__main__":\n    main("diffusion")\n', 'training/train_energy.py': 'from training.train_images import main\n\nif __name__ == "__main__":\n    main("energy")\n', 'scripts/assignment4_theory.py': '"""Execute all numerical/autograd questions and record the actual results."""\n\nimport json\nfrom pathlib import Path\nimport torch\n\n\ndef main():\n    result = {"torch_version": torch.__version__}\n    j = torch.arange(4, dtype=torch.float64)\n    angles = 10000 ** (-2 * j / 8)\n    result["Q2"] = torch.stack((angles.sin(), angles.cos()), 1).flatten().tolist()\n    x = torch.tensor([2.0], requires_grad=True)\n    (x**2 + 3 * x).backward()\n    result["Q6a"] = x.grad.tolist()\n    assert x.grad.item() == 7\n    x = torch.tensor([2.0])\n    y = x**2 + 3 * x\n    try:\n        y.backward()\n    except RuntimeError as error:\n        result["Q6b_error"] = str(error)\n    result["Q6c_and_Q7c_default"] = x.requires_grad\n    x = torch.tensor([2.0], requires_grad=True)\n    w = torch.tensor([1.0, 3.0])\n    (w[0] * x**2 + w[1] * x).backward()\n    result["Q7a"] = {\n        "x_grad": x.grad.tolist(),\n        "w_grad": None if w.grad is None else w.grad.tolist(),\n    }\n    x = torch.tensor([2.0], requires_grad=True)\n    w = torch.tensor([1.0, 3.0], requires_grad=True)\n    (w[0] * x**2 + w[1] * x).backward()\n    result["Q7b"] = {"x_grad": x.grad.tolist(), "w_grad": w.grad.tolist()}\n    assert w.grad.tolist() == [4.0, 2.0]\n    x = torch.tensor([1.0], requires_grad=True)\n    y = x * 3\n    z = y.detach()\n    w = z * 2\n    try:\n        w.backward()\n    except RuntimeError as error:\n        result["Q8_original_error"] = str(error)\n    x = torch.tensor([1.0], requires_grad=True)\n    y = x * 3\n    z = y\n    w = z * 2\n    w.backward()\n    result["Q8_fixed"] = x.grad.tolist()\n    assert x.grad.item() == 6\n    x = torch.tensor([1.0], requires_grad=True)\n    (x * 2).backward()\n    first = x.grad.item()\n    (x * 3).backward()\n    second = x.grad.item()\n    x.grad = None\n    (x * 3).backward()\n    result["Q9"] = {"first": first, "second": second, "after_reset": x.grad.item()}\n    assert (first, second, x.grad.item()) == (2.0, 5.0, 3.0)\n    output = (\n        Path(__file__).resolve().parents[1] / "results/assignment4/theory_results.json"\n    )\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'app/__init__.py': '"""Assignment 4 self-contained training package."""\n', 'app/image_generation/__init__.py': '"""Assignment 4 self-contained training package."""\n', 'training/__init__.py': '"""Assignment 4 self-contained training package."""\n'}
for name, source in SOURCE_FILES.items():
    p=WORK/name; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(source)
os.chdir(WORK)
if str(WORK) not in sys.path: sys.path.insert(0,str(WORK))
print(WORK)


## Train a DDPM
The model predicts sampled Gaussian noise. Only network weights are optimized. The cosine schedule uses 200 steps; inference uses EMA weights. This configuration starts a new larger model rather than resuming the supplied small checkpoint.

In [ ]:
GPU = torch.cuda.is_available()
BASE_CHANNELS = 32 if GPU else 8
DIFFUSION_EPOCHS = 50 if GPU else 3
ENERGY_EPOCHS = 30 if GPU else 3
BATCH_SIZE = 64
LANGEVIN_STEPS = 20 if GPU else 10
subprocess.run([sys.executable,"-u","-m","training.train_diffusion","--epochs",str(DIFFUSION_EPOCHS),"--base-channels",str(BASE_CHANNELS),"--batch-size",str(BATCH_SIZE)],check=True)


## Train an energy model
During sampling, network weights are frozen, image gradients are enabled, and each update is detached. The later contrastive backward updates weights using detached negative samples. Do not put the EBM sampler inside inference_mode. Its finite, clipped, small-noise chain is an approximation.

In [ ]:
subprocess.run([sys.executable,"-u","-m","training.train_energy","--epochs",str(ENERGY_EPOCHS),"--base-channels",str(BASE_CHANNELS),"--batch-size",str(BATCH_SIZE),"--langevin-steps",str(LANGEVIN_STEPS)],check=True)


## Inspect actual results
Training noise MSE and contrastive energy loss are different objectives and should not be compared numerically. Generated images, not loss alone, determine whether output quality is useful.

In [ ]:
from PIL import Image
from IPython.display import display
for kind in ["diffusion","energy"]:
    history=json.loads((WORK/f"results/assignment4/{kind}_training.json").read_text())
    print(kind, history[-1])
    display(Image.open(WORK/f"results/assignment4/{kind}_samples.png"))


## Export weights for the API
Replace the same filenames under the project models directory, then rebuild Docker. The API and deployment files are in the accompanying project.

In [ ]:
import zipfile
archive=WORK/"Assignment4_Trained_Models.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in (WORK/"models").glob("cifar10_*.pt"): z.write(p,"models/"+p.name)
    for p in (WORK/"results/assignment4").glob("*"):
        if p.is_file(): z.write(p,"results/assignment4/"+p.name)
print(archive)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))


## Run the gradient calculations

In [ ]:
subprocess.run([sys.executable,"scripts/assignment4_theory.py"],check=True)


# Assignment 4: Advanced Image Generation

Liuyang Li | November 11, 2026

## Part 2. Building Blocks of a Diffusion Model

### Question 1. Sinusoidal time embedding

Let the dimension d be even, M = 10000, and j = 0, ..., d/2 - 1. Define the frequencies and sine/cosine pairs:

\[
\omega_j=M^{-2j/d},\qquad e_{2j}(t)=\sin(t\omega_j),\qquad e_{2j+1}(t)=\cos(t\omega_j).
\]

Thus, the zero-indexed i-th coordinate is sin(t / M^(i/d)) when i is even, and cos(t / M^((i-1)/d)) when i is odd. Each pair shares a frequency. The angles are in radians. A learned MLP can subsequently transform this deterministic vector to condition the U-Net.

Convention: the calculations and model use sine-first interleaving with denominator d/2. Cosine-first grouped implementations reorder the same components. Implementations using d/2 - 1 instead change the actual frequencies, so an explicit convention is necessary.

### Question 2. Embedding dimension 8 and timestep 1

Substitute d = 8 and t = 1:

\[
\omega_j=10000^{-2j/8}=10000^{-j/4},\qquad (\omega_0,\omega_1,\omega_2,\omega_3)=(1,0.1,0.01,0.001).
\]

Since t = 1, these are also the angles.

| j | Angle | Sine | Cosine |
|---|---|---|---|
| 0 | 1 | 0.84147098 | 0.54030231 |
| 1 | 0.1 | 0.09983342 | 0.99500417 |
| 2 | 0.01 | 0.00999983 | 0.99995000 |
| 3 | 0.001 | 0.00100000 | 0.99999950 |

The interleaved embedding is:

```text
[0.84147098, 0.54030231, 0.09983342, 0.99500417,
 0.00999983, 0.99995000, 0.00100000, 0.99999950]
```

Reproducible calculation:

```python
j = torch.arange(4, dtype=torch.float64)
frequencies = 10000 ** (-2 * j / 8)
angles = 1.0 * frequencies
embedding = torch.stack((angles.sin(), angles.cos()), dim=1).flatten()
```

Cosine-first grouped order instead gives [0.54030231, 0.99500417, 0.99995000, 0.99999950, 0.84147098, 0.09983342, 0.00999983, 0.00100000] for these same frequencies.

### Question 3. Connection to transformer positional encoding

Both techniques represent a scalar index using sine and cosine functions at multiple frequencies. In the original transformer, the index is a token's position, and positional encodings are added to token embeddings so attention can use word order.

In diffusion, the index is the diffusion/noise timestep. Its embedding usually passes through an MLP and conditions several U-Net residual blocks through addition or feature modulation. The key difference is sequence position versus the global denoising timestep: a time embedding tells the denoiser which noise regime it must handle, rather than encoding each pixel's location.

### Question 4. Bottleneck resolution

Assume each stride-2 block halves each spatial dimension, as for a 4x4 convolution with padding 1:

\[
64\times64\longrightarrow32\times32\longrightarrow16\times16\longrightarrow8\times8.
\]

After three downsampling blocks, 64 / 2^3 = 8. The bottleneck spatial resolution is therefore 8x8. Its channel count depends on the architecture. The question concerns a hypothetical 64x64 input; the implemented CIFAR-10 generator uses native 32x32 images and a 4x4 bottleneck.

### Question 5. U-Net output and loss

In a noise-prediction DDPM, the U-Net receives the noisy image x_t and timestep t and outputs predicted Gaussian noise epsilon_theta(x_t, t). Its output has shape B x C x H x W, matching the input image. It is not a class label or scalar energy.

Define alpha_t = 1 - beta_t and alpha_bar_t as the product of alpha_s for s = 1 through t. Sample clean image x_0, timestep t, and epsilon from N(0, I), then construct:

\[
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon.
\]

The sampled noise is the known target. Compare it with the prediction:

\[
L_{\mathrm{simple}}=\mathbb{E}_{x_0,t,\epsilon}\left[\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2\right].
\]

The implementation averages squared errors over the batch and pixels. Backpropagation updates network parameters; the image, timestep and sampled noise are not optimizer parameters.

```python
t = torch.randint(T, (images.shape[0],), device=images.device)
noise = torch.randn_like(images)
a = alpha_bar[t][:, None, None, None]
x_t = a.sqrt() * images + (1 - a).sqrt() * noise
predicted_noise = unet(x_t, t)
loss = torch.nn.functional.mse_loss(predicted_noise, noise)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

Clean-image and velocity prediction are alternatives, but this implementation specifically predicts noise. Its array indices for diffusion steps start at zero.

## Part 3. Building Blocks of an Energy Model

### Question 6. Basic gradient calculations

(a) Differentiate y = x^2 + 3x:

\[
\frac{dy}{dx}=2x+3,\qquad \left.\frac{dy}{dx}\right|_{x=2}=2(2)+3=7.
\]

The code prints x.grad = tensor([7.]).

(b) With requires_grad=False on x, this particular expression has no input that requires gradients. Thus y.requires_grad is False and y.grad_fn is None. Calling y.backward() raises:

```text
RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
```

x.grad remains None. A different expression can still be tracked if another input, such as a trainable weight, requires gradients.

(c) torch.tensor defaults to requires_grad=False. Omitting the flag does not enable gradient tracking for the new tensor:

```python
x = torch.tensor([2.0])
print(x.requires_grad)  # False
```


### Question 7. Introduce weights

Write y = w_0 x^2 + w_1 x with x = 2, w_0 = 1 and w_1 = 3.

(a) In the supplied code, x requires gradients but w does not. The result is:

```text
x.grad = tensor([7.])
w.grad = None
```

PyTorch can use the numerical values of w to calculate dy/dx without accumulating derivatives into w. Calling a tensor a weight does not make it trainable. Standard neural network layer weights are normally nn.Parameter objects, which require gradients by default.

(b) The partial derivatives are:

\[
\frac{\partial y}{\partial w_0}=x^2=4,\qquad\frac{\partial y}{\partial w_1}=x=2.
\]

Enable gradients on w before the forward computation:

```python
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([1.0, 3.0], requires_grad=True)
y = w[0] * x**2 + w[1] * x
y.backward()
print("x.grad =", x.grad)
print("w.grad =", w.grad)
```

Output:

```text
x.grad = tensor([7.])
w.grad = tensor([4., 2.])
```

If only weight gradients are needed, x can have requires_grad=False; w still receives [4, 2].

(c) As in Question 6(c), a new torch.tensor defaults to requires_grad=False.

### Question 8. Breaking the graph

z = y.detach() disconnects z from the graph leading to y and x. Both z and w consequently do not require gradients, so w.backward() raises the error in Question 6(b). The numerical forward value remains 6; its gradient connection has been removed.

Retain z while removing detach:

```python
x = torch.tensor([1.0], requires_grad=True)
y = x * 3
z = y                  # z = y.clone() also preserves the connection
w = z * 2
w.backward()
print("x.grad =", x.grad)
```

By the chain rule:

\[
\frac{dw}{dx}=\frac{dw}{dz}\frac{dz}{dy}\frac{dy}{dx}=2\cdot1\cdot3=6.
\]

The corrected code prints x.grad = tensor([6.]). Setting z.requires_grad_(True) after detaching creates a new leaf but does not reconnect it to x.

Detaching between iterations is intentional in EBM sampling: each updated image becomes a new optimization variable and the full sampling trajectory need not be differentiated.

### Question 9. Gradient accumulation

PyTorch adds gradients into leaf tensors' .grad fields. The first backward contributes d(2x)/dx = 2. The second contributes d(3x)/dx = 3:

\[
g_1=2,\qquad g_2=g_1+3=2+3=5.
\]

The original code prints:

```text
After first backward: x.grad = tensor([2.])
After second backward: x.grad = tensor([5.])
```

Clear the gradient to obtain independent derivatives:

```python
x = torch.tensor([1.0], requires_grad=True)
y1 = x * 2
y1.backward()
print(x.grad)           # tensor([2.])
x.grad = None          # x.grad.zero_() is another option here

y2 = x * 3
y2.backward()
print(x.grad)           # tensor([3.])
```

In neural network training, optimizer.zero_grad(set_to_none=True) clears gradients of that optimizer's parameters. It does not clear an arbitrary input tensor unless the input belongs to the optimizer. Accumulation can be intentional when combining microbatches. Here the two expressions have separate forward graphs, so retain_graph=True is unnecessary and would not clear accumulated gradients.

## References

1. PyTorch, torch.tensor: https://docs.pytorch.org/docs/stable/generated/torch.tensor.html
2. PyTorch, Tensor.detach: https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html
3. PyTorch, Autograd mechanics: https://docs.pytorch.org/docs/stable/notes/autograd.html
4. Vaswani et al. (2017), Attention Is All You Need, Section 3.5: https://arxiv.org/abs/1706.03762
5. Ho, Jain and Abbeel (2020), Denoising Diffusion Probabilistic Models: https://arxiv.org/abs/2006.11239
6. OpenAI, guided-diffusion time embedding: https://github.com/openai/guided-diffusion/blob/main/guided_diffusion/nn.py
7. UvA, Deep Energy-Based Generative Models: https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial8/Deep_Energy_Models.html
